# 02 — Identity: Cognito, workload e adaptação para Entra

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a implantação atual exige quatro contas e as integrações Registry, Terraform e RAG, inclusive com `--labs`. A seleção parcial não converte esse runner em uma implantação de conta única. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 02](../docs/diagramas/svg/02_identity_entra_obo.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "02")
print({"lab": "02", "dependencies": list(dependencies), "region": config["region"]})

### Conectar ao ambiente implantado

Lemos os outputs exatos do CloudFormation. O cliente não cria infraestrutura e não substitui um serviço indisponível.

In [ ]:
lab = AWSWorkshop(CONFIG, domain="platform")
print(lab.preflight())

### Autenticar a pessoa no Cognito

O login usa USER_PASSWORD_AUTH para usuários sintéticos do workshop e SECRET_HASH para o cliente confidencial. Em uma aplicação de usuários finais, use authorization code com PKCE. O código abaixo mostra a chamada Cognito e a verificação de assinatura, emissor, audiência e token_use. Só imprimimos claims selecionadas.

In [ ]:
from santander_demo.aws_workshop.identity import login, verify
display(Code(inspect.getsource(login), language="python"))
display(Code(inspect.getsource(verify), language="python"))
token = lab.login("c", "supervisor")
claims = lab.claims(token)
print({key: claims[key] for key in ["iss", "sub", "aud", "token_use"]})

### Inspecionar o workload do agente

AgentCore Identity mantém uma identidade de workload separada da pessoa. O agente hospedado chama GetWorkloadAccessTokenForJWT usando sua role IAM e o JWT recebido. O WAT fica em memória no Runtime e não aparece nos outputs.

In [ ]:
from santander_demo.aws_workshop.agent import answer
environment = lab.outputs("agent")
control = lab.aws.client("bedrock-agentcore-control")
workload = control.get_workload_identity(name=environment["WorkloadName"])
print({"workloadArn": workload["workloadIdentityArn"]})
display(Code(inspect.getsource(answer), language="python"))

### Executar o agente hospedado com identidade e evidência

A inferência Bedrock e as chamadas de ferramentas executam no Runtime. O notebook envia a pergunta e verifica se houve uma chamada real de ferramenta.

In [ ]:
response = await lab.invoke_agent(token, "Consulte o risco de C001 e explique o resultado.")
assert response["workload_identity"] == "VERIFIED"
assert response["execution"] == "AWS_AGENTCORE_RUNTIME"
assert any(x["tool"] == "riscos___consultar_risco" for x in response["tool_calls"])
print(response)

**Cognito e Entra ID:** assinatura, emissor, audiência, expiração, identidade
humana e identidade da aplicação existem nos dois desenhos. Nesta jornada
Cognito, Gateway e Runtime confiam explicitamente na mesma audiência e
propagam o JWT para essa cadeia. Isso não é uma troca OBO.

No Entra, registre APIs/audiências próprias, configure o provider OAuth
do AgentCore Identity e o grant OBO suportado, e valide o token emitido
para cada destino. Microsoft Graph `/me` exige um token Graph delegado;
Cognito não emite esse token. O laboratório 09 demonstra separadamente M2M,
que representa uma aplicação e não herda permissões de uma pessoa.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.